In [5]:
import cv2
import openslide
import numpy as np

In [7]:
slide_dir = "/Volumes/Volumes/rsrch5/home/trans_mol_path/cercan/BE_master/0.input/flow/roi/flow/MDA12_ROI_pyramid/D0068_ROI1.tif"

In [ ]:
wsi = openslide.OpenSlide(slide_dir)
wsi.level_dimensions[0]

In [11]:

def generate_mask(slide, level=0):
    level = slide.get_best_level_for_downsample(64)
    image_ds = np.array(slide.read_region((0, 0), level, slide.level_dimensions[level]).convert('RGB'))
    img_gray = cv2.cvtColor(image_ds, cv2.COLOR_RGB2GRAY)
    img_blur = cv2.GaussianBlur(img_gray,(7,7),0)
    _, mask = cv2.threshold(img_blur, 30, 220, cv2.THRESH_OTSU+cv2.THRESH_BINARY_INV)

    original_dimensions = slide.level_dimensions[0] 
    mask = cv2.resize(mask, (original_dimensions[0], original_dimensions[1]), interpolation=cv2.INTER_NEAREST)

    kernel = np.ones((2, 2), np.uint8)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)


    largest_contour = max(contours, key=cv2.contourArea)
    x, y, w, h = cv2.boundingRect(largest_contour)
    
    clipped_image = image_ds[y:y+h, x:x+w]


    mask = np.array(mask)

    return clipped_image, mask



In [12]:
img, mask = generate_mask(wsi)

In [13]:
img.shape

(0, 0, 3)